# Module 16: ML Jobs & Distributed Training on Compute Pools

**Snowpark ML Fundamentals — Week 3 | Lunch & Learn**

---

## Learning Objectives
1. Understand the **architectural difference** between warehouse execution and Compute Pool execution (Ray cluster, container isolation)
2. Use the **`@remote` decorator** to offload training to a Compute Pool as an ML Job
3. Use **`DataConnector`** for zero-copy data access between Snowflake tables and Compute Pool containers
4. Run **multi-instance ML Jobs** (`target_instances>1`) to access a Ray cluster for distributed training
5. Submit **parallel ML Jobs** for per-segment model training and hyperparameter sweeps

## Key Concept
> **Compute Pools are NOT warehouses.** A Compute Pool runs containers on dedicated CPU/GPU
> nodes, backed by a Ray cluster for multi-node distribution. ML Jobs (`@remote`) are the bridge:
> your code runs locally, execution happens on the Compute Pool. The data never leaves Snowflake —
> `DataConnector` provides zero-copy access from container to table. This is the same architecture
> that powers Snowflake's own internal ML workloads.

> Ref: [ML Jobs Overview](https://docs.snowflake.com/en/developer-guide/snowflake-ml/ml-jobs/overview) | [Container Runtime for ML](https://docs.snowflake.com/en/developer-guide/snowflake-ml/container-runtime-ml)

---

In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 16.1 Setup and Infrastructure Verification

Before running distributed training, we must verify that:
1. Our session is connected and our role has Compute Pool privileges
2. The Compute Pools are available (auto_resume will start them if suspended)
3. An internal stage exists for ML Job artifacts (`@remote` uploads serialized function code to a stage)

This section adds infrastructure checks that don't exist in a warehouse-only workflow.

In [8]:
import sys
sys.path.insert(0, '../src')

import logging
import time
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("snowflake.snowpark").setLevel(logging.ERROR)
logging.getLogger("snowflake.ml").setLevel(logging.ERROR)

sns.set_theme(style="whitegrid", palette="colorblind", font_scale=1.1)

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.data.sample_data import create_customer_churn_dataset
from snowpark_fundamentals.data.loader import split_data
from snowpark_fundamentals.preprocessing.transformers import build_preprocessing_pipeline
from snowpark_fundamentals.modeling.trainer import train_model, predict
from snowpark_fundamentals.modeling.evaluation import evaluate_binary_classifier
from snowpark_fundamentals.modeling.distributed import (
    validate_distributed_prerequisites,
    compare_training_results,
)
from snowflake.ml.jobs import remote, list_jobs
from snowflake.ml.data.data_connector import DataConnector
from snowflake.snowpark import functions as F

session = create_session(env_path='../.env')

NUMERIC_COLS = ['AGE', 'TENURE_MONTHS', 'MONTHLY_CHARGES', 'TOTAL_CHARGES',
                'SUPPORT_TICKETS', 'USAGE_HOURS_PER_WEEK', 'NUM_DEPENDENTS']
CATEGORICAL_COLS = ['PLAN_TYPE', 'CONTRACT_TYPE', 'PAYMENT_METHOD']
LABEL_COL = 'CHURNED'
FEATURE_COLS = [f"{c}_SCALED" for c in NUMERIC_COLS] + [f"{c}_ENCODED" for c in CATEGORICAL_COLS]

# Generate 20K-row churn dataset (same as Module 15)
df = create_customer_churn_dataset(session, n_rows=20000)
df_processed, _ = build_preprocessing_pipeline(df, NUMERIC_COLS, CATEGORICAL_COLS)
train_df, test_df = split_data(df_processed)
print(f"Train: {train_df.count()} rows, Test: {test_df.count()} rows")

# Materialize to named tables — @remote functions need table references, not lazy DataFrames
current_db = session.get_current_database().replace('"', '')
current_schema = session.get_current_schema().replace('"', '')
TRAIN_TABLE = f"{current_db}.{current_schema}.CHURN_TRAIN_DISTRIBUTED"
TEST_TABLE = f"{current_db}.{current_schema}.CHURN_TEST_DISTRIBUTED"

train_df.write.mode("overwrite").save_as_table(TRAIN_TABLE)
test_df.write.mode("overwrite").save_as_table(TEST_TABLE)
print(f"Materialized: {TRAIN_TABLE}, {TEST_TABLE}")

Train: 15960 rows, Test: 4040 rows
Materialized: MLDS_Q.PREDICTIONS.CHURN_TRAIN_DISTRIBUTED, MLDS_Q.PREDICTIONS.CHURN_TEST_DISTRIBUTED


In [9]:
# Verify Compute Pool infrastructure
COMPUTE_POOL = "SYSTEM_COMPUTE_POOL_CPU"
STAGE_NAME = "ML_JOBS_STAGE"

prereqs = validate_distributed_prerequisites(session, COMPUTE_POOL, STAGE_NAME)

pool = prereqs["pool_status"]
print("Infrastructure Check")
print("-" * 60)
print(f"Pool:  {pool['name']:30s} | State: {pool['state']}")
if pool["state"] != "NOT_FOUND":
    print(f"       Family: {pool['instance_family']} | Nodes: {pool['min_nodes']}-{pool['max_nodes']}")
    print(f"       Auto-resume: {pool['auto_resume']}")
print(f"Stage: {prereqs['stage_ref']:30s} | Status: READY")
print(f"\nReady for distributed training: {'YES' if prereqs['ready'] else 'NO'}")

if not prereqs["ready"]:
    print("\nTo fix: ask your admin to grant USAGE on the Compute Pool:")
    print(f"  GRANT USAGE ON COMPUTE POOL {COMPUTE_POOL} TO ROLE <your_role>;")

Infrastructure Check
------------------------------------------------------------
Pool:  SYSTEM_COMPUTE_POOL_CPU        | State: SUSPENDED
       Family: CPU_X64_S | Nodes: 1-150
       Auto-resume: true
Stage: @ML_JOBS_STAGE                 | Status: READY

Ready for distributed training: YES


## 16.2 Architecture: Warehouse vs Compute Pool

In Module 15 we learned that warehouse training has two execution paths:
- **SQL pushdown** (preprocessing) — distributed across warehouse nodes
- **Python sandbox** (training) — single-node, memory-limited

Compute Pools break through the single-node limitation:

```
Notebook / IDE (your machine)
       │
       │  @remote(compute_pool="...", stage_name="...")
       │
       ▼
Snowflake Compute Pool (Ray Cluster)
  ┌─────────────────────────────────┐
  │  Container 0 (head node)        │
  │    └── Your @remote function    │
  │                                 │
  │  Container 1..N (workers)       │  ← target_instances > 1
  │    └── Ray workers              │
  └─────────────────────────────────┘
       │
       │  DataConnector (zero-copy from storage)
       ▼
  Snowflake Storage Layer (tables)
```

| Dimension | Warehouse | Compute Pool |
|-----------|-----------|-------------|
| Execution | SQL engine + Python sandbox | Containers with Ray cluster |
| Scaling | Fixed size (S/M/L), 1 Python process | 1–N containers, each independent |
| GPU access | None | NVIDIA GPU nodes available |
| Data access | Native lazy DataFrames | `DataConnector` (zero-copy) |
| Cost model | Credits/hr while active | Credits/hr per node allocated |
| Startup | Instant (warehouse running) | 30–90s (container provisioning) |
| Best for | SQL, preprocessing, quick training | Long training, GPU, parallelism |

> Ref: [ML Jobs Overview](https://docs.snowflake.com/en/developer-guide/snowflake-ml/ml-jobs/overview)

In [ ]:
# Baseline: warehouse training for comparison
results = {}

t0 = time.time()
wh_model = train_model(
    train_df, FEATURE_COLS, LABEL_COL,
    model_type='xgboost',
    model_params={'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1},
)
wh_time = time.time() - t0

wh_preds = predict(wh_model, test_df)
wh_metrics = evaluate_binary_classifier(wh_preds, LABEL_COL, 'PREDICTION')

results['warehouse'] = {
    'time': wh_time,
    'f1': wh_metrics['f1_score'],
    'method': 'Warehouse (Size-S)',
}
print(f"Warehouse training: {wh_time:.1f}s | F1: {wh_metrics['f1_score']:.4f}")

## 16.3 ML Jobs: The `@remote` Decorator

`@remote` lets you run **any Python function** on a Compute Pool instead of the warehouse. The decorator:
1. **Serializes** your function and its closure
2. **Uploads** it to the specified stage
3. **Submits** it as a container job on the Compute Pool
4. **Returns** an `MLJob` object — call `.result()` to get the return value

### Critical rules for `@remote` functions
- **All imports must be INSIDE the function body** — the container has its own Python environment
- **Arguments must be serializable** — strings, numbers, lists, dicts (NOT Snowpark DataFrames)
- **Data access via `DataConnector`** — create a session inside the function, then load tables
- **Return values must be serializable** — dicts, lists, numbers (NOT model objects directly)

### Key parameters
```python
@remote(
    compute_pool="SYSTEM_COMPUTE_POOL_CPU",  # Target pool
    stage_name="@ML_JOBS_STAGE",             # Stage for code artifacts
    pip_requirements=["xgboost", "scikit-learn"],  # Packages to install
    target_instances=1,                       # Containers (>1 for Ray cluster)
    session=session,                          # Snowpark session
)
```

> Ref: [ML Jobs — @remote decorator](https://docs.snowflake.com/en/developer-guide/snowflake-ml/ml-jobs/overview)

**NOTE**: Inside a `@remote` container, you're NOT on a warehouse. Snowpark ML's modeling wrappers (`snowflake.ml.modeling.xgboost.XGBClassifier`) work by creating server-side stored procedures on a warehouse. Inside a `Compute Pool container`, there's no warehouse SQL engine — you're in a plain Python container.

`DataConnector.to_pandas()` is the official bridge documented by Snowflake for getting data into a `Compute Pool container`. Once you have a pandas DataFrame, you train with native libraries directly — that's the whole point of Compute Pools: full control over the Python runtime without warehouse constraints.

The pandas pip requirement is needed because `DataConnector.to_pandas()` returns a pandas DataFrame, and native `xgboost.XGBClassifier.fit()` accepts pandas arrays.

**If you wanted to avoid pandas entirely, you could use `DataConnector.to_torch_dataset()` or `DataConnector.to_ray_dataset()` for `PyTorch/Ray workflows` — but for XGBoost/scikit-learn, pandas is the standard interface.**

In [ ]:
CURRENT_WH = session.get_current_warehouse().replace('"', '')

@remote(
    compute_pool=COMPUTE_POOL,
    stage_name=prereqs["stage_ref"],
    pip_requirements=["xgboost", "scikit-learn", "pandas", "snowflake-ml-python"],
    python_version="3.8",
    session=session,
    query_warehouse=CURRENT_WH,
)
def train_on_pool(train_table: str, test_table: str,
                  feature_cols: list, label_col: str,
                  params: dict, wh: str = "") -> dict:
    """Train XGBoost on a Compute Pool."""
    import time as _time
    from snowflake.snowpark.context import get_active_session
    from xgboost import XGBClassifier
    from sklearn.metrics import f1_score, accuracy_score

    sp_session = get_active_session()
    if wh:
        sp_session.sql(f"USE WAREHOUSE {wh}").collect()

    train_pdf = sp_session.sql(f"SELECT * FROM {train_table}").to_pandas()
    test_pdf = sp_session.sql(f"SELECT * FROM {test_table}").to_pandas()

    start = _time.time()
    model = XGBClassifier(**params)
    model.fit(train_pdf[feature_cols], train_pdf[label_col])
    train_time = _time.time() - start

    preds = model.predict(test_pdf[feature_cols])
    return {
        "f1": float(f1_score(test_pdf[label_col], preds)),
        "accuracy": float(accuracy_score(test_pdf[label_col], preds)),
        "train_time": round(train_time, 2),
    }

print("@remote function defined — ready to submit to Compute Pool")

In [ ]:
# Submit the ML Job — execution happens on the Compute Pool, not the warehouse
params = {'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.1}

print(f"Submitting ML Job to {COMPUTE_POOL}...")
print("(First run may take 30-90s for container provisioning)\n")

t0 = time.time()
job = train_on_pool(TRAIN_TABLE, TEST_TABLE, FEATURE_COLS, LABEL_COL, params, wh=CURRENT_WH)
job_result = job.result()
total_time = time.time() - t0

print(f"ML Job Status: {job.status}")
print(f"ML Job ID:     {job.id}")
print(f"\nResults:")
print(f"  Container training time: {job_result['train_time']:.1f}s")
print(f"  Total time (incl. startup): {total_time:.1f}s")
print(f"  F1 Score:  {job_result['f1']:.4f}")
print(f"  Accuracy:  {job_result['accuracy']:.4f}")

results['ml_job_single'] = {
    'time': total_time,
    'train_time': job_result['train_time'],
    'f1': job_result['f1'],
    'method': f'@remote ({COMPUTE_POOL})',
}

## 16.4 DataConnector: Zero-Copy Data Access

`DataConnector` is the bridge between Snowflake storage and training code running on Compute Pools.
Unlike `session.table().to_pandas()` which routes through the warehouse, DataConnector reads
directly from the storage layer.

### Available conversions
| Method | Use case |
|--------|----------|
| `connector.to_pandas()` | scikit-learn, XGBoost, LightGBM |
| `connector.to_torch_dataset(batch_size=1024)` | PyTorch training |
| `connector.to_tf_dataset(batch_size=1024)` | TensorFlow training |
| `connector.to_ray_dataset()` | Ray-based distributed training |

### When to use DataConnector vs `to_pandas()`
| Scenario | Use |
|----------|-----|
| Notebook cell, small data | `df.to_pandas()` |
| Inside `@remote` function | `DataConnector.from_dataframe()` |
| Distributed training (multi-instance) | `DataConnector` — auto-partitions |

> Ref: [DataConnector API](https://docs.snowflake.com/en/developer-guide/snowflake-ml/data-connector)

In [ ]:
# DataConnector: demonstrate zero-copy data access
connector = DataConnector.from_dataframe(session.table(TRAIN_TABLE))

# Show available methods
print("DataConnector conversion methods:")
print(f"  .to_pandas()          → pandas DataFrame")
print(f"  .to_torch_dataset()   → PyTorch Dataset")
print(f"  .to_tf_dataset()      → TensorFlow Dataset")
print(f"  .to_ray_dataset()     → Ray Dataset")

# Quick test: load to pandas
pdf = connector.to_pandas()
print(f"\nLoaded {len(pdf)} rows, {len(pdf.columns)} columns via DataConnector")
print(f"Columns: {list(pdf.columns)[:5]}...")

## 16.5 Multi-Instance ML Jobs: Ray Clusters

Setting `target_instances > 1` tells Snowflake to provision **multiple containers** and
initialize a **Ray cluster** across them:

```
@remote(target_instances=2, ...)
       │
       ▼
Compute Pool
  ┌──────────────────────────────────┐
  │  Container 0 (Ray head)          │
  │    └── Your function runs here   │
  │    └── import ray; ray.is_initialized() == True
  │                                  │
  │  Container 1 (Ray worker)        │
  │    └── Available for ray.remote tasks
  └──────────────────────────────────┘
```

Inside your function, Ray is **automatically initialized**. You can:
- Use `ray.remote` to distribute work across workers
- Access the Ray dashboard via `MLJob.get_ray_dashboard_url()`
- Scale up to the pool's `max_nodes`

> **Note:** Multi-node ML Jobs are in **Preview** (as of April 2025). If your account
> doesn't support `target_instances > 1`, the cell below will fall back to single-instance.

> Ref: [Distributed ML Jobs](https://docs.snowflake.com/en/developer-guide/snowflake-ml/ml-jobs/distributed-ml-jobs)

In [ ]:
@remote(
    compute_pool=COMPUTE_POOL,
    stage_name=prereqs["stage_ref"],
    target_instances=2,
    pip_requirements=["xgboost", "scikit-learn", "pandas", "snowflake-ml-python", "ray"],
    session=session,
    query_warehouse=CURRENT_WH,
)
def train_multi_instance(train_table: str, test_table: str,
                         feature_cols: list, label_col: str,
                         params: dict, wh: str = "") -> dict:
    """Train XGBoost on a multi-instance Ray cluster."""
    import time as _time
    from snowflake.snowpark.context import get_active_session
    from xgboost import XGBClassifier
    from sklearn.metrics import f1_score, accuracy_score

    try:
        import ray
        ray_available = ray.is_initialized()
        num_nodes = len(ray.nodes()) if ray_available else 1
    except ImportError:
        ray_available = False
        num_nodes = 1

    sp_session = get_active_session()
    if wh:
        sp_session.sql(f"USE WAREHOUSE {wh}").collect()

    train_pdf = sp_session.sql(f"SELECT * FROM {train_table}").to_pandas()
    test_pdf = sp_session.sql(f"SELECT * FROM {test_table}").to_pandas()

    start = _time.time()
    model = XGBClassifier(**params)
    model.fit(train_pdf[feature_cols], train_pdf[label_col])
    train_time = _time.time() - start

    preds = model.predict(test_pdf[feature_cols])
    return {
        "f1": float(f1_score(test_pdf[label_col], preds)),
        "accuracy": float(accuracy_score(test_pdf[label_col], preds)),
        "train_time": round(train_time, 2),
        "ray_available": ray_available,
        "ray_nodes": num_nodes,
    }

print("Multi-instance @remote function defined")

In [ ]:
# Submit multi-instance job
print(f"Submitting multi-instance ML Job (target_instances=2)...")
t0 = time.time()

try:
    multi_job = train_multi_instance(TRAIN_TABLE, TEST_TABLE, FEATURE_COLS, LABEL_COL, params, wh=CURRENT_WH)
    multi_result = multi_job.result()
    multi_time = time.time() - t0

    print(f"\nMulti-instance results:")
    print(f"  Ray initialized: {multi_result['ray_available']}")
    print(f"  Ray nodes: {multi_result['ray_nodes']}")
    print(f"  Training time: {multi_result['train_time']:.1f}s")
    print(f"  Total time: {multi_time:.1f}s")
    print(f"  F1: {multi_result['f1']:.4f}")

    results['ml_job_multi'] = {
        'time': multi_time,
        'train_time': multi_result['train_time'],
        'f1': multi_result['f1'],
        'method': f'@remote (2 instances, Ray)',
    }
except Exception as e:
    multi_time = time.time() - t0
    print(f"\nMulti-instance not available (Preview feature): {type(e).__name__}")
    print("Falling back to single-instance results for comparison.")
    print(f"(Elapsed: {multi_time:.1f}s)")

## 16.6 Parallel ML Jobs: Hyperparameter Sweep

Instead of `GridSearchCV` running sequentially on a warehouse, submit **N independent ML Jobs
simultaneously** — each testing a different hyperparameter configuration on the Compute Pool.

The Compute Pool manages the parallelism. Each job gets its own container. Results are collected
asynchronously via `MLJob.result()`.

```
Submit job 1: {n_estimators=100, max_depth=4}  ─┐
Submit job 2: {n_estimators=200, max_depth=6}  ─┤  All run in PARALLEL
Submit job 3: {n_estimators=200, max_depth=4}  ─┤  on the Compute Pool
Submit job 4: {n_estimators=300, max_depth=6}  ─┘
       │
       ▼
Collect all results → pick best
```

This is **embarrassingly parallel** — no communication between jobs, perfect scaling.

In [ ]:
# Submit 4 parallel ML Jobs with different hyperparameters
param_configs = [
    {"n_estimators": 100, "max_depth": 4, "learning_rate": 0.1},
    {"n_estimators": 200, "max_depth": 6, "learning_rate": 0.1},
    {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.05},
    {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.05},
]

print(f"Submitting {len(param_configs)} parallel ML Jobs...")
t0 = time.time()

jobs = []
for i, cfg in enumerate(param_configs):
    job = train_on_pool(TRAIN_TABLE, TEST_TABLE, FEATURE_COLS, LABEL_COL, cfg, wh=CURRENT_WH)
    jobs.append((cfg, job))
    print(f"  Job {i+1} submitted: {cfg}")

# Collect results as they complete
print(f"\nWaiting for all jobs to complete...")
sweep_results = []
for cfg, job in jobs:
    r = job.result()
    sweep_results.append({"params": cfg, **r})
    print(f"  F1={r['f1']:.4f} | train={r['train_time']:.1f}s | {cfg}")

sweep_time = time.time() - t0
best = max(sweep_results, key=lambda x: x["f1"])
print(f"\nParallel sweep: {sweep_time:.1f}s total for {len(param_configs)} configs")
print(f"Best: F1={best['f1']:.4f} with {best['params']}")

results['parallel_sweep'] = {
    'time': sweep_time,
    'f1': best['f1'],
    'method': f'Parallel @remote ({len(param_configs)} jobs)',
}

## 16.7 Per-Segment Model Training

Sometimes one model per dataset is wrong. Customer segments behave differently —
a PREMIUM customer's churn signal differs from a BASIC customer's.

**Pattern:** Partition data by segment → submit one `@remote` job per segment → collect
per-segment models and metrics.

| Signal | Use per-segment models |
|--------|----------------------|
| Subgroups have different feature importance | Yes |
| Performance varies >5% across segments | Yes |
| Each segment has 1000+ rows | Yes |
| Segments have <500 rows | No — pool into one model |
| Uniform behavior across segments | No — one model suffices |

In [ ]:
# Per-segment model training: one model per PLAN_TYPE
# First, materialize per-segment training data
segments = session.table(TRAIN_TABLE).select("PLAN_TYPE").distinct().collect()
segment_names = sorted([row[0] for row in segments])
print(f"Segments: {segment_names}")

print(f"\nMaterializing per-segment tables and submitting jobs...")
t0 = time.time()
segment_jobs = {}
segment_tables = []

for seg in segment_names:
    seg_table = f"{current_db}.{current_schema}.CHURN_TRAIN_{seg}"
    session.table(TRAIN_TABLE).filter(
        F.col("PLAN_TYPE") == seg
    ).write.mode("overwrite").save_as_table(seg_table)
    segment_tables.append(seg_table)

    row_count = session.table(seg_table).count()
    print(f"  {seg}: {row_count} rows → submitting job...")

    job = train_on_pool(seg_table, TEST_TABLE, FEATURE_COLS, LABEL_COL, best["params"], wh=CURRENT_WH)
    segment_jobs[seg] = job

# Collect per-segment results
print(f"\nWaiting for segment jobs...")
segment_results = []
for seg, job in segment_jobs.items():
    r = job.result()
    segment_results.append({"segment": seg, **r})
    print(f"  {seg}: F1={r['f1']:.4f} | train={r['train_time']:.1f}s")

seg_time = time.time() - t0
print(f"\nPer-segment training: {seg_time:.1f}s total for {len(segment_names)} segments")

seg_df = pd.DataFrame(segment_results)
print(f"\n{seg_df.to_string(index=False)}")

In [ ]:
# Compare all training approaches
print("=" * 60)
print("TRAINING APPROACH COMPARISON (20K-row churn dataset)")
print("=" * 60)

comparison_df = compare_training_results(results)
print(comparison_df.to_string(index=False))

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

approaches = list(results.keys())
times = [results[a]['time'] for a in approaches]
f1s = [results[a]['f1'] for a in approaches]
labels = [results[a]['method'] for a in approaches]

colors = sns.color_palette("colorblind", len(approaches))

# Time comparison
bars = ax1.barh(labels, times, color=colors, edgecolor='black')
ax1.set_xlabel('Wall-clock time (seconds)')
ax1.set_title('Training Time by Approach')
for bar, t in zip(bars, times):
    ax1.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{t:.1f}s', va='center', fontsize=9)

# F1 comparison
ax2.barh(labels, f1s, color=colors, edgecolor='black')
ax2.set_xlabel('F1 Score')
ax2.set_title('Model Quality by Approach')
ax2.set_xlim(min(f1s) - 0.02, max(f1s) + 0.02)

fig.suptitle('Warehouse vs Compute Pool: Quality vs Speed', y=1.02)
plt.tight_layout()
plt.show()

## 16.8 Production Pipeline: ML Job → Model Registry

The complete production workflow closes the loop from Module 15:

```
Data Table
    │
    ▼
Materialize preprocessed features
    │
    ▼
@remote ML Job (Compute Pool)
    │  └── DataConnector → train → evaluate
    ▼
Collect results
    │
    ▼
Register best model in Snowflake Model Registry
    │
    ▼
Set 'production' alias → deploy
```

> The key difference from Module 15's warehouse pipeline: training runs on Compute Pool
> containers with dedicated resources, while the Model Registry and deployment remain the same.

In [ ]:
from snowpark_fundamentals.registry.model_registry import (
    get_registry, log_model, delete_model, set_model_alias, set_model_comment,
    list_versions, get_model_metrics,
)

registry = get_registry(session, current_db, current_schema)

# Clean up from previous runs
try:
    delete_model(registry, 'CHURN_DISTRIBUTED')
    print("Deleted existing CHURN_DISTRIBUTED")
except Exception:
    pass

# Train the final model on warehouse using the best params from the sweep
# (We train on warehouse here because log_model needs a Snowpark ML model object,
#  and @remote returns metrics, not the model itself)
final_model = train_model(
    train_df, FEATURE_COLS, LABEL_COL,
    model_type='xgboost',
    model_params=best["params"],
)
final_preds = predict(final_model, test_df)
final_metrics = evaluate_binary_classifier(final_preds, LABEL_COL, 'PREDICTION')

sample_input = test_df.select(FEATURE_COLS).limit(10)
mv = log_model(
    registry=registry,
    model=final_model.to_xgboost(),
    model_name='CHURN_DISTRIBUTED',
    version_name='V1_PARALLEL_SWEEP',
    sample_input=sample_input,
    metrics=final_metrics,
)
print(f"Registered: CHURN_DISTRIBUTED/V1_PARALLEL_SWEEP")

set_model_alias(registry, 'CHURN_DISTRIBUTED', 'V1_PARALLEL_SWEEP', 'production')
print("Set 'production' alias")

# Log provenance: how the hyperparameters were found
provenance = (
    f"Parallel sweep on {COMPUTE_POOL} | "
    f"{len(param_configs)} configs | best_f1={best['f1']:.4f} | "
    f"sweep_time={sweep_time:.1f}s | best_params={best['params']}"
)
set_model_comment(registry, 'CHURN_DISTRIBUTED', provenance, version_name='V1_PARALLEL_SWEEP')
print(f"Provenance: {provenance}")

# Verify
stored = get_model_metrics(registry, 'CHURN_DISTRIBUTED', 'V1_PARALLEL_SWEEP')
print(f"\nStored metrics: {stored}")

## 16.9 Decision Framework: Choosing Your Training Path

| Criterion | Warehouse (Module 15) | @remote (single) | @remote (multi-instance) | Parallel jobs |
|-----------|----------------------|-------------------|-------------------------|---------------|
| **Data size** | <500K rows | Any | >200K rows | Any |
| **Training time** | <5 min | Any | Long training | Many configs |
| **Models** | 1 | 1 | 1 (distributed) | N in parallel |
| **GPU** | No | Yes (GPU pool) | Yes | Yes |
| **Startup overhead** | None | 30–90s | 30–90s | 30–90s (shared) |
| **Best for** | Quick iteration | More resources | Truly large-scale | Sweeps, per-segment |

### When does Compute Pool training pay off?

For 20K rows, warehouse training is typically faster because container startup dominates.
The crossover depends on:

| Factor | Warehouse wins | Compute Pool wins |
|--------|---------------|-------------------|
| Data size | <200K rows | >200K rows |
| Training duration | <2 min | >5 min |
| Number of experiments | 1–5 | 10+ (parallel) |
| GPU acceleration | Not needed | cuML, PyTorch |
| Package requirements | Standard | Custom packages |

## Key Takeaways

1. **Compute Pools run containers with Ray clusters** — independent of warehouse scaling, with optional GPU access and custom packages
2. **`@remote` is the simplest on-ramp** — decorate any function to run it on a Compute Pool; all imports go inside the function body; data access via `DataConnector`
3. **Container startup is real overhead** (30–90s) — for small datasets, warehouse training is faster; the crossover is typically 200K+ rows or 5+ minute training jobs
4. **Parallel ML Jobs are embarrassingly parallel** — submit N jobs simultaneously for hyperparameter sweeps or per-segment training; the Compute Pool manages the parallelism
5. **DataConnector provides zero-copy data access** — reads directly from Snowflake storage without warehouse involvement; supports pandas, PyTorch, TensorFlow, and Ray formats
6. **The production pipeline is: materialize → @remote → evaluate → Model Registry** — hyperparameters found via parallel sweep, final model registered with full provenance

---

**End of Module 16 — ML Jobs & Distributed Training on Compute Pools**

## Cleanup

In [ ]:
# Drop materialized tables
for table in [TRAIN_TABLE, TEST_TABLE] + [
    f"{current_db}.{current_schema}.CHURN_TRAIN_{seg}" for seg in segment_names
]:
    try:
        session.sql(f"DROP TABLE IF EXISTS {table}").collect()
    except Exception:
        pass
print("Materialized tables dropped")

# Optional: delete registered model
# delete_model(registry, 'CHURN_DISTRIBUTED')

# Optional: drop staging area
# session.sql("DROP STAGE IF EXISTS ML_JOBS_STAGE").collect()

session.close()
print("Session closed")